# Pandas 파일별 1 · 00 Foundations 표 살펴보기 · 고르기 · 계산

**한 줄 요약** · 6명 학생 표로 DataFrame의 모양 · 열 · 행 · 조건 · 계산 · 정렬 · 개수를 익히고, 처음 보는 상품 CSV에 그대로 적용

- **원본** · `5. pandas/01_학습노트/00_pandas_foundations.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/04_Pandas` 전체 정리 1 ~ 4장 (표 살펴보기 · 선택하기 · 조건 · 계산과 정렬)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `shape` · `columns` · `dtypes` · `head` · `tail`로 표의 크기와 자료형을 먼저 확인한다
2. 열 하나는 Series, 열 목록은 DataFrame임을 알고 `loc`(라벨) · `iloc`(위치)로 행 · 열을 고른다
3. 조건으로 True/False(mask)를 만들고, 조건 두 개는 괄호와 `&` · `|`로 묶어 행을 고른다
4. `mean` · `sum` · 새 열 · `sort_values` · `value_counts`로 질문에 숫자로 답한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **CSV를 처음 읽었을 때** · `shape` · `dtypes` · `head` (1장)
- **특성 열 X와 정답 열 y를 나눌 때** · `df[[...]]` · `df["label"]` (2-1)
- **학습용 행 · 특정 클래스만 고를 때** · 조건 mask · `&amp;` (3장)
- **클래스 분포 · 실험 결과 순위를 볼 때** · `value_counts` · `sort_values` (4장)

## 학습 지도

<img src="images/pd1_map.png" alt="Foundations 학습 지도" width="763">

**예상 학습 시간** · 개념 50분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>라이브러리 · 출력 설정 · 원본 문제</td></tr>
<tr><td>1</td><td>표 만들고 살펴보기</td><td>10만 행을 세지 않고 표의 크기 · 열 · 자료형을 어떻게 확인하는가?</td></tr>
<tr><td>2</td><td>열 · 행 고르기</td><td>필요한 열 · 행만 이름과 위치로 어떻게 고르는가?</td></tr>
<tr><td>3</td><td>조건으로 찾기</td><td>조건에 맞는 행만 고르려면 어떤 두 단계를 거치는가?</td></tr>
<tr><td>4</td><td>계산 · 새 열 · 정렬 · 개수</td><td>골라낸 데이터로 통계 · 새 정보 · 순위 · 개수를 어떻게 만드는가?</td></tr>
<tr><td>5</td><td>종합 실습</td><td>처음 보는 CSV에서 1 ~ 4장 방법을 스스로 꺼내 쓰려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

pandas를 불러오고 한글 열 정렬을 켬. 종합 실습이 읽는 `data/products_basic.csv`도 확인. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · `pandas` (실행 결과는 pandas 3 기준 · 문자열 열 dtype이 `str`, 2.x는 `object`)
- **seed · 장치** · 필요 없음 (무작위 · GPU 없음)
- **데이터** · `students` 6명 · `sales` 3개 (코드 안에서 만듦) · 종합 실습 `data/products_basic.csv` 13개
- **빈칸 표시** · 코드의 `# 빈칸 00-①` = 원본 노트북의 첫 번째 빈 코드 칸 (모두 13칸)

**실행 흐름**

1. `Path` · `pandas` 불러오기
2. 한글 폭 출력 설정
3. pandas 버전 · 실습 CSV 존재 확인

In [1]:
# 라이브러리 · 출력 설정 · 데이터 폴더 확인
# 왜 · 한글 열이 어긋나지 않게 출력 폭을 맞추고, 실습 CSV가 있는지 먼저 확인하기 위해
from pathlib import Path  # 파일 경로

import pandas as pd  # 관례: pd 로 줄여 부름

pd.set_option("display.unicode.east_asian_width", True)  # 한글을 2칸으로 세어 열 맞춤
print("pandas:", pd.__version__)                         # 설치된 버전 기록
print("data/products_basic.csv 있음:", Path("data/products_basic.csv").exists())  # 종합 실습 파일

pandas: 3.0.3
data/products_basic.csv 있음: True


### 원본 문제

- **첫 질문** · 데이터가 6행이면 눈으로 확인할 수 있지만, 10만 행이면 어떻게 할까?
- **오늘 할 일** · 살펴보기 → 선택하기 → 조건으로 찾기 → 계산하기 → 정렬하기
- **목표** · 함수 이름을 외우는 것이 아니라 질문에 맞는 데이터를 직접 찾기
- **빈칸** · 빈 코드 칸 13개 (크기 · 마지막 행 · 열 선택 · loc/iloc · 조건 4개 · 계산 · 새 열 · 정렬 · 개수)
- **글 빈칸** · "이 데이터는 몇 행 몇 열?" · "조건을 만들면 각 행마다 ___가 만들어지고 ___인 행만 선택"

**원본 힌트**

- 크기는 `shape`, 열 이름은 `columns`
- 마지막 행들은 `tail()`
- 이름 · 라벨 기준이면 `loc`, 몇 번째인지 위치 기준이면 `iloc`
- 조건 하나를 True/False로 만든 뒤 대괄호에 넣기 · 두 조건은 각각 괄호로 묶음

---
## 1. 표 만들고 살펴보기

> **이 장의 흐름**
>
> - 원본 1 · 2장: 학생 6명 표를 만들고 크기 · 열 · 자료형 · 앞뒤 행을 확인
> - 큰 데이터도 같은 네 가지로 먼저 파악

<img src="images/pandas_dataframe.png" alt="DataFrame = index · 열 이름 · 열마다 자료형" width="659">

**파이프라인 위치** · ① 데이터 준비 — 표의 모양 확인

### 1-1. `A` DataFrame 만들기와 크기 · `pd.DataFrame()` · `shape` · `columns` <sub>원본 2장 · 빈칸 00-①</sub>

> 🎯 **학생 6명 × 항목 4개 표의 크기와 열 이름은 어떻게 확인하는가?**

💡 **쉽게 말하면**

- DataFrame은 **엑셀 시트 한 장**.
- 사전의 키가 열 이름, 값 목록이 그 열의 값이 됨.
- `shape`는 시트의 **(행 수, 열 수)**.

**실행 흐름**

1. 사전 → DataFrame
2. 표 출력
3. shape · columns · len · size

**먼저 예측** · 원본 글 빈칸 · 이 데이터는 몇 행 몇 열? — 답은 코드 실행 결과 아래에

In [2]:
# 학생 6명 표 만들기: {열 이름: 값 목록}
# 왜 · 파일 없이도 결과를 눈으로 맞춰 볼 수 있는 작은 표를 직접 만들기 위해
students = pd.DataFrame({  # 사전의 키 = 열 이름, 값 목록 = 그 열의 값
    "name": ["민수", "지수", "영희", "철수", "유진", "현우"],
    "score": [78, 95, 84, 65, 91, 88],
    "city": ["서울", "부산", "서울", "대전", "서울", "부산"],
    "study_hours": [3, 7, 5, 2, 6, 5],
})
print(students)            # 표 전체를 눈으로 확인


# 빈칸 00-① 크기와 열 이름
# 왜 · 처음 보는 표는 크기와 열 이름부터 확인해 잘못 읽은 곳을 찾기 위해
print("shape:", students.shape)                         # (행 수, 열 수)
print("columns:", students.columns.tolist())            # 열 이름 목록
print("len:", len(students), "/ size:", students.size)  # 행 수 · 칸 수

   name  score  city  study_hours
0  민수     78  서울            3
1  지수     95  부산            7
2  영희     84  서울            5
3  철수     65  대전            2
4  유진     91  서울            6
5  현우     88  부산            5
shape: (6, 4)
columns: ['name', 'score', 'city', 'study_hours']
len: 6 / size: 24


**결과** · 예측 정답은 6행 4열 — `(6, 4)`

- `민수 78 서울 3` — 왼쪽 0 ~ 5는 자동 번호(index)
- `shape: (6, 4)` — 원본 글 빈칸 답: 행 6 · 열 4
- `columns: ['name', 'score', 'city', 'study_hours']` — 열 이름 목록 (오타 확인용)
- `len: 6 / size: 24` — len = 행 수 · size = 6 × 4 칸

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df.shape</code></td><td><strong>(행 수, 열 수)</strong></td><td>크기를 처음 확인할 때 (기본 선택)</td></tr>
<tr><td><code>len(df)</code></td><td><strong>행 수</strong>만</td><td>반복 · 비율 계산</td></tr>
<tr><td><code>df.size</code></td><td>칸 수 = 행 × 열</td><td>거의 안 씀</td></tr>
<tr><td><code>df.columns</code></td><td>열 이름 목록</td><td>열 이름 오타 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `shape`는 괄호 없는 속성 → `df.shape()`는 TypeError

> ✅ **한 줄 정리** · `df.shape`로 (행 수, 열 수)를, `df.columns`로 열 이름을 먼저 확인한다.

**딥러닝 코드에서는**

- `print(train_df.shape)` — 학습 샘플 수 확인
- `df.columns`로 특성 · 라벨 열 이름 확인

### 1-2. `A` 자료형과 앞뒤 일부 보기 · `dtypes` · `head()` · `tail()` <sub>원본 2장 · 빈칸 00-②</sub>

> 🎯 **점수 `88`과 도시 `"부산"`을 pandas는 같은 종류로 보는가?**

💡 **쉽게 말하면**

- 열마다 **한 가지 규격 상자**를 씀.
- 큰 표는 전부 출력하지 않고 앞 · 뒤 몇 줄만 맛보기.

**실행 흐름**

1. dtypes
2. head(3)
3. 원본 tail(2)
4. 빈칸 tail(3)

**먼저 예측** · `head(3)`은 몇 행을 보여 줄까? — 답은 코드 실행 결과 아래에

In [3]:
# 열마다 자료형
# 왜 · 숫자로 계산할 열이 문자열로 읽히지 않았는지 확인하기 위해
print(students.dtypes)  # 숫자 열인가 문자열 열인가


# 처음 3행 · 원본 tail(2)
# 왜 · 전체를 출력하지 않고 앞 · 뒤 일부로 내용을 확인하기 위해
print(students.head(3))  # 기본값 5 → 3으로 지정
print(students.tail(2))  # 마지막 2행


# 빈칸 00-② 마지막 3행
print(students.tail(3))  # 3 · 4 · 5번 행

name             str
score          int64
city             str
study_hours    int64
dtype: object
   name  score  city  study_hours
0  민수     78  서울            3
1  지수     95  부산            7
2  영희     84  서울            5
   name  score  city  study_hours
4  유진     91  서울            6
5  현우     88  부산            5
   name  score  city  study_hours
3  철수     65  대전            2
4  유진     91  서울            6
5  현우     88  부산            5


**결과** · 예측 정답은 3행 — 위치 0 · 1 · 2

- `score int64` — 점수 열은 정수 → 계산 가능
- `name str` — 문자열 열 (pandas 3은 `str`, 2.x는 `object`로 표시)
- `2 영희 84 서울 5` — `head(3)`의 마지막 = 위치 2
- `3 철수 65 대전 2` — 빈칸 00-② 답: `tail(3)`은 3 · 4 · 5번 행

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df.dtypes</code></td><td>열마다 자료형만</td><td>계산 전 숫자 열인지 확인</td></tr>
<tr><td><code>df.info()</code></td><td>자료형 + 결측 아닌 개수</td><td>처음 받은 파일 점검</td></tr>
<tr><td><code>df.head(n)</code></td><td><strong>앞</strong> n행 (기본 5)</td><td>값 형식 눈으로 보기</td></tr>
<tr><td><code>df.tail(n)</code></td><td><strong>뒤</strong> n행</td><td>끝에 붙은 이상한 행 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 숫자처럼 보여도 dtype이 `str`이면 계산 불가 → `dtypes` 먼저 확인

> ✅ **한 줄 정리** · `dtypes`로 열마다 자료형을, `head()` · `tail()`로 앞뒤 일부만 보고 표를 파악한다.

**딥러닝 코드에서는**

- 라벨 CSV를 읽은 뒤 `df.head()` · `df.dtypes`로 경로 · 라벨 형식 확인

---
## 2. 열 · 행 고르기

> **이 장의 흐름**
>
> - 1장에서 표 전체를 파악
> - 원본 3 · 4장: 열은 `[]`, 행 · 열 함께는 `loc` · `iloc`

<img src="images/pandas_loc_iloc.png" alt="정렬 뒤 loc은 라벨 0, iloc은 첫 줄" width="644">

**파이프라인 위치** · ② 불러오기 — 특성 열 · 정답 열 고르기

### 2-1. `A` 열 고르기 · Series와 DataFrame · `df["열"]` · `df[["열"]]` <sub>원본 3장 · 빈칸 00-③ · ④</sub>

> 🎯 **한 열을 고르면 Series, 두 열을 고르면 DataFrame이 되는 이유는?**

💡 **쉽게 말하면**

- Series는 **이름표 붙은 한 줄 목록**.
- DataFrame은 그 목록 여러 개를 나란히 붙인 표.
- 대괄호 안에 **목록**을 넣으면 표가 나옴.

**실행 흐름**

1. score 한 열
2. type 비교
3. 빈칸: city · name과 city
4. 빈칸: type · shape

**먼저 예측** · `students[["name", "city"]]`의 type은? — 답은 코드 실행 결과 아래에

In [4]:
# 한 열 → Series
# 왜 · 대괄호 하나로 고른 결과가 1차원 Series라는 것을 확인하기 위해
score = students["score"]                             # 열 이름 하나
print(type(students).__name__, type(score).__name__)  # 표 vs 한 줄 목록
print(score.tolist())                                 # 점수 6개


# 빈칸 00-③ city 한 열 · name과 city 두 열
# 왜 · 괄호 하나와 이중 괄호의 결과 모양 차이를 비교하기 위해
city = students["city"]                 # Series
name_city = students[["name", "city"]]  # 열 이름 목록 → DataFrame
print(name_city.head(2))                # 두 열이 표로 나옴


# 빈칸 00-④ 두 결과의 type · shape
# 1차원 vs 2차원
print(type(city).__name__, city.shape, "/", type(name_city).__name__, name_city.shape)

DataFrame Series
[78, 95, 84, 65, 91, 88]
   name  city
0  민수  서울
1  지수  부산
Series (6,) / DataFrame (6, 2)


**결과** · 예측 정답은 DataFrame — 안쪽 대괄호가 열 이름 목록

- `DataFrame Series` — 표 전체 vs 한 열
- `[78, 95, 84, 65, 91, 88]` — Series의 값 6개
- `0 민수 서울` — 빈칸 00-③ 답: 두 열이 표로 나옴
- `Series (6,) / DataFrame (6, 2)` — 빈칸 00-④ 답: 1차원 vs 2차원

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df["열"]</code></td><td>열 하나 → <strong>Series</strong> (1차원)</td><td>한 열로 계산 · 비교</td></tr>
<tr><td><code>df[["열1", "열2"]]</code></td><td>열 목록 → <strong>DataFrame</strong></td><td>표 모양 유지</td></tr>
<tr><td><code>df[["열"]]</code></td><td>한 열이어도 목록이면 DataFrame</td><td>모델 입력처럼 2차원이 필요할 때</td></tr>
<tr><td><code>df["a", "b"]</code></td><td>목록 없이 쉼표 → KeyError</td><td>쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 여러 열은 이중 대괄호 `df[["a", "b"]]` → 쉼표만 쓰면 KeyError

> ✅ **한 줄 정리** · 열 이름 하나면 Series, 열 이름 목록이면 DataFrame이 된다.

**딥러닝 코드에서는**

- `X = df[["age", "income"]]` (특성 표) · `y = df["label"]` (정답 Series)

### 2-2. `A` 라벨과 위치로 고르기 · `loc` · `iloc` <sub>원본 4장 · 빈칸 00-⑤</sub>

> 🎯 **`loc`과 `iloc`은 무엇을 기준으로 찾고, 정렬 뒤에는 결과가 어떻게 달라지는가?**

💡 **쉽게 말하면**

- `loc`은 **이름표(라벨)로 찾기**.
- `iloc`은 **앞에서 몇 번째 자리로 찾기**.
- 정렬하면 이름표는 사람을 따라가고, 자리는 새로 매겨짐.

**실행 흐름**

1. loc · iloc 기본
2. 빈칸: 첫 학생 · 처음 3명 · 첫 2행 2열
3. 정렬 뒤 loc[0] vs iloc[0]

**먼저 예측** · `students.loc[0:2]`와 `students.iloc[0:2]`는 각각 몇 행? — 답은 코드 실행 결과 아래에

In [5]:
# loc: 라벨 기준 (끝 라벨 포함) · iloc: 위치 기준 (끝 위치 미포함)
# 왜 · 같아 보이는 두 방법이 끝 포함 여부에서 다르다는 것을 확인하기 위해
print(students.loc[0, "name"], students.iloc[0, 0])  # 지금은 라벨 = 위치라 같음
print(students.loc[0:2, ["name", "score"]].shape)    # 라벨 0 · 1 · 2 → 3행
print(students.iloc[0:3, 0:2].shape)                 # 위치 0 · 1 · 2 → 3행


# 빈칸 00-⑤ 첫 학생 · 처음 3명 이름 · 점수 · 첫 2행 2열
# 왜 · 행 · 열을 함께 고르는 대표 모양을 손에 익히기 위해
print(students.loc[0].to_dict())             # 라벨 0 행 전체
print(students.loc[0:2, ["name", "score"]])  # 처음 3명
print(students.iloc[:2, :2])                 # 첫 2행 · 첫 2열


# 정렬하면 라벨과 위치가 달라짐
# 왜 · 정렬 뒤에는 loc와 iloc가 다른 행을 가리킨다는 함정을 보이기 위해
ranked = students.sort_values("score", ascending=False)  # 라벨은 사람을 따라감
print("라벨 순서:", ranked.index.tolist())               # 첫 줄 라벨은 1
print("loc[0] →", ranked.loc[0, "name"], "/ iloc[0] →", ranked.iloc[0]["name"])  # 라벨 0 vs 첫 줄

민수 민수
(3, 2)
(3, 2)
{'name': '민수', 'score': 78, 'city': '서울', 'study_hours': 3}
   name  score
0  민수     78
1  지수     95
2  영희     84
   name  score
0  민수     78
1  지수     95
라벨 순서: [1, 4, 5, 2, 0, 3]
loc[0] → 민수 / iloc[0] → 지수


**결과** · 예측 정답은 3행 · 2행 — `loc`은 끝 라벨 포함, `iloc`은 끝 위치 미포함

- `{'name': '민수', 'score': 78, 'city': '서울', 'study_hours': 3}` — 빈칸 00-⑤ 답: 라벨 0 행 전체
- `2 영희 84` — `loc[0:2, ...]`는 0 · 1 · 2번 3행 (끝 포함)
- `라벨 순서: [1, 4, 5, 2, 0, 3]` — 정렬해도 라벨은 원래 번호를 들고 다님
- `loc[0] → 민수 / iloc[0] → 지수` — 라벨 0(민수) vs 지금 첫 줄(1등 지수)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df.loc[라벨, 열이름]</code></td><td><strong>라벨</strong> 기준 · 끝 라벨 <strong>포함</strong></td><td>행 이름 · 조건 · 열 이름으로</td></tr>
<tr><td><code>df.iloc[위치, 위치]</code></td><td><strong>위치</strong> 기준 · 끝 위치 <strong>미포함</strong></td><td>첫 n행처럼 순서로</td></tr>
<tr><td>정렬 뒤 <code>loc[0]</code></td><td>라벨 0을 찾음 (첫 줄 아님)</td><td>원래 행을 찾을 때</td></tr>
<tr><td>정렬 뒤 <code>iloc[0]</code></td><td>지금 <strong>첫 줄</strong></td><td>1등이 필요할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 필터 · 정렬 뒤 `loc[0]`은 첫 줄이 아님 → 라벨 0이 없으면 KeyError

> ✅ **한 줄 정리** · `loc`은 라벨 기준(끝 포함), `iloc`은 위치 기준(끝 미포함)으로 행 · 열을 고른다.

**딥러닝 코드에서는**

- `df.iloc[:800]` — 앞 800행을 학습용으로
- `df.loc[df["split"] == "val", "path"]` — 조건 + 열

---
## 3. 조건으로 찾기

> **이 장의 흐름**
>
> - 2장은 위치 · 이름을 알 때
> - 원본 5장: 조건 → 행마다 True/False → True 행만

<img src="images/pandas_mask.png" alt="조건 → True/False → True 행만" width="748">

**파이프라인 위치** · ② 불러오기 · ③ 전처리 — 학습용 행 · 특정 클래스

### 3-1. `A` 조건 하나로 고르기 · `df[mask]` <sub>원본 5장 · 빈칸 00-⑥ · ⑦</sub>

> 🎯 **80점 이상인 학생만 고르려면 어떤 두 단계를 거치는가?**

💡 **쉽게 말하면**

- 조건은 **행마다 O · X를 적은 출석부**(mask).
- 표에 출석부를 대면 O인 줄만 남음.

**실행 흐름**

1. 조건 → mask
2. df[mask]
3. 빈칸: 85점 이상 · 부산
4. True 개수

**먼저 예측** · `students["score"] >= 80`의 결과는 숫자일까, 이름일까, True/False일까? — 답은 코드 실행 결과 아래에

In [6]:
# 조건 → 행마다 True/False (mask)
# 왜 · 조건식이 행마다 True/False를 만든다는 것을 먼저 눈으로 보기 위해
mask = students["score"] >= 80          # 6개 답
print("mask:", mask.tolist())           # 민수 · 철수만 False
print(students[mask]["name"].tolist())  # True 행의 이름


# 빈칸 00-⑥ · ⑦ 85점 이상 · 부산 학생
# 왜 · mask를 대괄호에 넣어 조건에 맞는 행만 고르기 위해
print("85점 이상:", students[students["score"] >= 85]["name"].tolist())  # 조건 → 고르기
print("부산:", students[students["city"] == "부산"]["name"].tolist())    # 문자열 비교
print("85점 이상 몇 명:", (students["score"] >= 85).sum())               # True = 1로 세기

mask: [False, True, True, False, True, True]
['지수', '영희', '유진', '현우']
85점 이상: ['지수', '유진', '현우']
부산: ['지수', '현우']
85점 이상 몇 명: 3


**결과** · 예측 정답은 True/False 6개 — 원본 글 빈칸 답은 각 행마다 **True/False**, 그중 **True**인 행만 선택

- `mask: [False, True, True, False, True, True]` — 학생마다 하나 · 민수 · 철수만 False
- `85점 이상: ['지수', '유진', '현우']` — 빈칸 00-⑥ 답
- `부산: ['지수', '현우']` — 빈칸 00-⑦ 답 · 문자열 비교도 같은 흐름
- `85점 이상 몇 명: 3` — True를 1로 세면 개수

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df["score"] &gt;= 80</code></td><td>행마다 True/False (mask)</td><td>조건을 먼저 눈으로</td></tr>
<tr><td><code>df[mask]</code></td><td>True 행만 남은 표</td><td>조건에 맞는 행 고르기 (기본 선택)</td></tr>
<tr><td><code>df.loc[mask, ["열"]]</code></td><td>조건 행 + 원하는 열</td><td>고른 뒤 열까지 줄일 때</td></tr>
<tr><td><code>mask.sum()</code></td><td>True 개수</td><td>몇 명인지만 필요할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 고른 결과에 바로 값을 넣으면 원본이 안 바뀜 → `df.loc[mask, "열"] = 값`

> ✅ **한 줄 정리** · 조건식으로 행마다 True/False를 만든 뒤 `df[mask]`로 True인 행만 고른다.

**딥러닝 코드에서는**

- `train_df = df[df["split"] == "train"]`
- `df[df["label"] != -1]` — 라벨 없는 행 제외

### 3-2. `A` 조건 묶기 · `&` · `|` · `isin()` <sub>원본 5장 · 빈칸 00-⑧ · ⑨</sub>

> 🎯 **조건 두 개를 함께 쓸 때 왜 `&` · `|`와 괄호가 필요한가?**

💡 **쉽게 말하면**

- `and`는 **질문 하나에 예/아니오 하나**만 답함.
- mask는 행마다 답이 있어서 줄마다 답하는 `&` · `|`가 필요.
- `&`가 `>=`보다 먼저 계산되므로 괄호로 순서를 정함.

**실행 흐름**

1. 원본: 서울 & 80↑ · 서울 | 부산
2. 빈칸: 부산 & 85↑
3. isin과 비교
4. 괄호 없음 에러

**먼저 예측** · 서울 또는 부산 학생은 몇 명? — 답은 코드 실행 결과 아래에

In [7]:
# 원본 예시: 서울 그리고 80점 이상 · 서울 또는 부산
# 왜 · 두 조건을 &와 |로 묶는 방법을 비교하기 위해
seoul_80 = (students["city"] == "서울") & (students["score"] >= 80)        # 조건마다 괄호 + &
seoul_busan = (students["city"] == "서울") | (students["city"] == "부산")  # 또는 = |
print("서울 & 80↑:", students[seoul_80]["name"].tolist())                  # 둘 다 True
print("서울 | 부산:", students[seoul_busan]["name"].tolist())              # 하나라도 True


# 빈칸 00-⑧ 부산 & 85점 이상 · ⑨ 서울 또는 부산 (isin)
# 왜 · 값 여러 개 중 하나인지 볼 때 isin이 | 보다 짧다는 것을 보이기 위해
busan_85 = (students["city"] == "부산") & (students["score"] >= 85)  # 그리고
print("부산 & 85↑:", students[busan_85]["name"].tolist())            # 지수 · 현우
# | 여러 번 = isin
print("isin 같은가:", students["city"].isin(["서울", "부산"]).equals(seoul_busan))


# 괄호를 빼면 & 가 먼저 계산됨
# 왜 · 연산자 우선순위 때문에 조건마다 괄호가 필요하다는 것을 에러로 확인하기 위해
try:
    students[students["city"] == "서울" & students["score"] >= 80]  # "서울" & score 부터
except TypeError:                                                   # 문자열과 & 계산 불가
    print("괄호 없음 → TypeError")                                  # 괄호가 필요한 이유

서울 & 80↑: ['영희', '유진']
서울 | 부산: ['민수', '지수', '영희', '유진', '현우']
부산 & 85↑: ['지수', '현우']
isin 같은가: True
괄호 없음 → TypeError


**결과** · 예측 정답은 5명 — 대전의 철수만 빠짐

- `서울 & 80↑: ['영희', '유진']` — 서울 3명 중 민수(78)만 제외
- `서울 | 부산: ['민수', '지수', '영희', '유진', '현우']` — 빈칸 00-⑨ 답
- `부산 & 85↑: ['지수', '현우']` — 빈칸 00-⑧ 답
- `괄호 없음 → TypeError` — `"서울" & students["score"]`부터 계산됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>(a) &amp; (b)</code></td><td><strong>둘 다</strong> True</td><td>그리고 · 범위</td></tr>
<tr><td><code>(a) &#124; (b)</code></td><td><strong>하나라도</strong> True</td><td>또는</td></tr>
<tr><td><code>~(a)</code></td><td>True ↔ False</td><td>조건 반대</td></tr>
<tr><td><code>isin([값1, 값2])</code></td><td><code>&#124;</code>를 여러 번 쓴 것과 같음</td><td>같은 열의 여러 값 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 괄호를 빼면 에러 또는 **에러 없이 다른 조건** → 조건마다 항상 괄호

> ✅ **한 줄 정리** · 여러 조건은 `&` · `|`로 묶고, 각 조건은 반드시 괄호로 감싼다.

**딥러닝 코드에서는**

- `df[(df["split"] == "train") & (df["label"].isin([0, 1]))]`

---
## 4. 계산 · 새 열 · 정렬 · 개수

> **이 장의 흐름**
>
> - 3장에서 원하는 행을 고름
> - 원본 6 ~ 9장: 통계 → 새 열 → 정렬 → 값별 개수

<img src="images/pandas_new_column.png" alt="열끼리 계산은 같은 행끼리" width="718">

**파이프라인 위치** · ③ 전처리 · ⑥ 평가 — 경로 · 라벨 열, 분포, 순위

### 4-1. `B` 기본 통계 · `mean()` · `sum()` · `agg()` <sub>원본 6장 · 빈칸 00-⑩</sub>

> 🎯 **점수를 하나씩 더하지 않고 평균 · 최대 · 합계를 구하려면?**

💡 **쉽게 말하면**

- 열 하나(Series)에 **계산 버튼**이 달려 있음.
- 누르면 그 열 전체를 계산.

**실행 흐름**

1. 원본: 점수 통계 4개
2. 빈칸: 공부 시간 통계 3개
3. agg로 한 표

**먼저 예측** · 점수 평균은 80보다 클까? — 답은 코드 실행 결과 아래에

In [8]:
# 원본: 점수의 평균 · 최댓값 · 최솟값 · 합계
# 왜 · 반복문 없이 열 전체 통계를 바로 구하기 위해
s = students["score"]  # 한 열 = Series
print("평균:", s.mean(), "/ 최대:", s.max(), "/ 최소:", s.min(), "/ 합계:", s.sum())  # 계산 버튼


# 빈칸 00-⑩ 공부 시간 평균 · 최댓값 · 합계
h = students["study_hours"]  # 공부 시간 열
# 소수 둘째 자리
print("공부 시간 평균:", round(h.mean(), 2), "/ 최대:", h.max(), "/ 합계:", h.sum())


# 여러 통계를 한 번에
# 왜 · 여러 열 · 여러 통계를 agg 한 번으로 표로 받기 위해
print(students[["score", "study_hours"]].agg(["mean", "max", "sum"]).round(2))  # 행 = 통계

평균: 83.5 / 최대: 95 / 최소: 65 / 합계: 501
공부 시간 평균: 4.67 / 최대: 7 / 합계: 28
      score  study_hours
mean   83.5         4.67
max    95.0         7.00
sum   501.0        28.00


**결과** · 예측 정답은 크다 — `83.5`

- `평균: 83.5 / 최대: 95 / 최소: 65 / 합계: 501` — 501 ÷ 6 = 83.5
- `공부 시간 평균: 4.67 / 최대: 7 / 합계: 28` — 빈칸 00-⑩ 답 · 28 ÷ 6 = 4.67
- `mean 83.5 4.67` — `agg`는 행 = 통계, 열 = 원래 열

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>s.mean()</code> · <code>s.sum()</code></td><td>통계 <strong>하나</strong></td><td>숫자 하나가 필요할 때</td></tr>
<tr><td><code>df.agg(["mean", "max"])</code></td><td>열마다 · 통계마다 <strong>표</strong></td><td>여러 통계 비교</td></tr>
<tr><td><code>df.describe()</code></td><td>개수 · 평균 · 사분위수 전부</td><td>처음 보는 숫자 열</td></tr>
<tr><td>Python <code>sum(s)</code></td><td>NaN 처리가 다르고 느림</td><td>쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 문자열 열에 `.mean()`은 TypeError → 숫자 열만 골라 계산

> ✅ **한 줄 정리** · 한 열은 `.mean()` · `.sum()`으로, 여러 통계는 `.agg([...])`로 한 번에 구한다.

**딥러닝 코드에서는**

- `df["label"].mean()` — 양성 비율
- `train_df[cols].agg(["mean", "std"])` — 정규화 통계

### 4-2. `A` 새 열 만들기 · `df["새열"] =` <sub>원본 6 · 7장 · 빈칸 00-⑪</sub>

> 🎯 **가격과 수량만 있을 때 할인 금액 열을 어떻게 만드는가?**

💡 **쉽게 말하면**

- 열끼리 계산하면 **같은 줄끼리 짝지어** 계산됨.
- 결과를 `df["새 이름"]`에 넣으면 새 열이 추가.

**실행 흐름**

1. passed 열
2. 판매 표 · sales = price × quantity
3. discount 목록
4. 빈칸: discount_amount

**먼저 예측** · 원본 질문 · passed 열을 만든 뒤 열은 몇 개? — 답은 코드 실행 결과 아래에

In [9]:
# 원본: 80점 이상이면 True인 passed 열
# 왜 · 비교 결과를 새 열로 저장해 뒤에서 개수 · 비율을 세기 위해
students["passed"] = students["score"] >= 80  # 비교 결과를 새 열로
print(students.shape)                         # 열이 5개가 됨


# 원본 판매 데이터: 가격 × 수량 → sales 열
# 왜 · 같은 행끼리 계산한 값을 새 열로 만드는 방법을 보이기 위해
sales = pd.DataFrame({"product": ["키보드", "마우스", "노트"],  # 상품 3개
                      "price": [50000, 25000, 3000],
                      "quantity": [1, 2, 3]})
sales["sales"] = sales["price"] * sales["quantity"]             # 같은 행끼리 곱
sales["discount"] = [0.1, 0.0, 0.2]                             # 행 수와 같은 길이의 목록


# 빈칸 00-⑪ 할인 금액 = 할인 전 금액 × 할인율
# 왜 · 앞에서 만든 열을 다시 계산에 쓰는 흐름을 연습하기 위해
sales["discount_amount"] = sales["sales"] * sales["discount"]  # 새 열
print(sales)                                                   # 열 6개 확인

(6, 5)
  product  price  quantity  sales  discount  discount_amount
0  키보드  50000         1  50000       0.1           5000.0
1  마우스  25000         2  50000       0.0              0.0
2    노트   3000         3   9000       0.2           1800.0


**결과** · 예측 정답은 5개 — `(6, 5)`

- `(6, 5)` — 원래 4열 + passed
- `0 키보드 50000 1 50000 0.1 5000.0` — 빈칸 00-⑪ 답: 50000 × 0.1 = 5000
- `1 마우스 25000 2 50000 0.0 0.0` — 25000 × 2 = 50000 · 할인 없음
- `2 노트 3000 3 9000 0.2 1800.0` — 9000 × 0.2 = 1800

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df["새열"] = 열 계산</code></td><td>같은 행끼리 계산해 <strong>원본에 추가</strong></td><td>새 정보 만들기 (기본 선택)</td></tr>
<tr><td><code>df["새열"] = [목록]</code></td><td>행 수와 <strong>길이가 같아야</strong> 함</td><td>값을 직접 적을 때</td></tr>
<tr><td><code>df.assign(새열=...)</code></td><td>원본은 두고 <strong>새 표</strong></td><td>여러 단계를 이어 쓸 때</td></tr>
<tr><td>계산만 출력</td><td>표는 그대로</td><td>저장 전 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 같은 이름으로 다시 저장하면 경고 없이 덮어씀

> ✅ **한 줄 정리** · `df["새열"] = 열 계산`으로 같은 행끼리 계산한 결과를 새 열로 저장한다.

**딥러닝 코드에서는**

- `df["path"] = root + "/" + df["file"]` — 이미지 경로 열

### 4-3. `B` 정렬 · `sort_values()` <sub>원본 8장 · 빈칸 00-⑫</sub>

> 🎯 **점수가 높은 순서로 보고, 동점이면 다른 기준을 쓰려면?**

💡 **쉽게 말하면**

- `sort_values`는 **기준 열로 줄 세우기**.
- 기본은 작은 값부터, `ascending=False`면 큰 값부터.
- 동점이면 두 번째 기준을 줌.

**실행 흐름**

1. 원본: 오름 · 내림차순
2. 빈칸: 공부 시간 긴 순 (동점은 점수)

**먼저 예측** · 공부 시간 5시간인 영희(84)와 현우(88) 중 누가 먼저? — 답은 코드 실행 결과 아래에

In [10]:
# 원본: 점수 오름차순 · 내림차순
# 왜 · ascending 하나로 정렬 방향이 바뀌는 것을 비교하기 위해
asc = students.sort_values("score")                    # 작은 값부터
desc = students.sort_values("score", ascending=False)  # 큰 값부터
print("오름차순:", asc["name"].tolist())               # 65 → 95
print("내림차순:", desc["name"].tolist())              # 95 → 65


# 빈칸 00-⑫ 공부 시간이 긴 학생부터 (같으면 점수 높은 순)
# 왜 · 동점일 때 순서를 정하려고 두 번째 기준 열을 함께 주기 위해
by_hours = students.sort_values(["study_hours", "score"], ascending=False)  # 동점은 두 번째 기준
print(by_hours[["name", "study_hours", "score"]])                           # 정렬 결과

오름차순: ['철수', '민수', '영희', '현우', '유진', '지수']
내림차순: ['지수', '유진', '현우', '영희', '민수', '철수']
   name  study_hours  score
1  지수            7     95
4  유진            6     91
5  현우            5     88
2  영희            5     84
0  민수            3     78
3  철수            2     65


**결과** · 예측 정답은 현우 — 두 번째 기준 점수가 더 높음

- `오름차순: ['철수', '민수', '영희', '현우', '유진', '지수']` — 65 → 95
- `내림차순: ['지수', '유진', '현우', '영희', '민수', '철수']` — 95 → 65
- `5 현우 5 88` — 빈칸 00-⑫ 답: 동점 5시간 → 점수 88이 먼저

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>sort_values("열")</code></td><td>오름차순</td><td>낮은 것부터</td></tr>
<tr><td><code>sort_values("열", ascending=False)</code></td><td>내림차순</td><td>높은 것부터 (순위)</td></tr>
<tr><td><code>sort_values(["열1", "열2"])</code></td><td>열1이 같으면 열2로</td><td>동점 순서를 정할 때</td></tr>
<tr><td><code>nlargest(n, "열")</code></td><td>정렬 + <code>head(n)</code>과 같은 값</td><td>상위 n개만</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 기준이 하나면 동점끼리의 순서는 보장되지 않음 → 두 번째 기준 열 추가

> ✅ **한 줄 정리** · `sort_values(열, ascending=False)`로 큰 값부터 정렬하고, 동점은 열 목록으로 정한다.

**딥러닝 코드에서는**

- `results.sort_values("val_loss").head(1)` — 가장 좋은 실험

### 4-4. `A` 값별 개수 · `value_counts()` <sub>원본 9장 · 빈칸 00-⑬</sub>

> 🎯 **도시마다 학생이 몇 명인지 한 번에 세려면?**

💡 **쉽게 말하면**

- `value_counts`는 **투표함 개표**.
- 같은 값끼리 모아 몇 표인지 세고 많은 순으로 보여 줌.

**실행 흐름**

1. 원본: 도시별 수
2. 빈칸: 합격 · 불합격 수
3. 비율

**먼저 예측** · 서울 · 부산 · 대전 학생은 각각 몇 명? — 답은 코드 실행 결과 아래에

In [11]:
# 원본: 도시별 학생 수
# 왜 · 범주마다 몇 개인지 한 줄로 세기 위해
print(students["city"].value_counts().to_dict())  # 값별 개수 (많은 순)


# 빈칸 00-⑬ 합격 · 불합격 학생 수
# 왜 · 개수뿐 아니라 비율(normalize)도 함께 보기 위해
print("passed:", students["passed"].value_counts().to_dict())  # True 4 · False 2
print("비율:", students["passed"].value_counts(normalize=True).round(2).to_dict())  # 개수 대신 비율

{'서울': 3, '부산': 2, '대전': 1}
passed: {True: 4, False: 2}
비율: {True: 0.67, False: 0.33}


**결과** · 예측 정답은 3 · 2 · 1명

- `{'서울': 3, '부산': 2, '대전': 1}` — 많은 순서
- `passed: {True: 4, False: 2}` — 빈칸 00-⑬ 답
- `비율: {True: 0.67, False: 0.33}` — `normalize=True` → 합이 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>value_counts()</code></td><td>값별 개수 · 많은 순 · NaN 제외</td><td>범주 분포 (기본 선택)</td></tr>
<tr><td><code>value_counts(dropna=False)</code></td><td>NaN도 한 줄</td><td>결측까지 볼 때</td></tr>
<tr><td><code>value_counts(normalize=True)</code></td><td>비율</td><td>클래스 불균형 확인</td></tr>
<tr><td><code>groupby(열).size()</code></td><td>같은 개수 · 값 순서</td><td>다른 집계와 함께 (보강1 Merge 권 4장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 기본은 NaN을 세지 않음 → 결측까지 보려면 `dropna=False`

> ✅ **한 줄 정리** · `value_counts()`로 값별 개수를, `normalize=True`로 비율을 구한다.

**딥러닝 코드에서는**

- `df["label"].value_counts(normalize=True)` — 클래스 불균형 확인

---
## 5. 종합 실습 · 처음 보는 상품 CSV

> **이 장의 흐름**
>
> - 원본 마지막 말: 다음에는 처음 보는 작은 CSV에서 같은 기술을 스스로 사용
> - 같은 폴더 `data/products_basic.csv`로 1 ~ 4장을 한 번에

<img src="images/pd1_practice.png" alt="상품 13개에서 조건 · 정렬 · 개수로 찾은 답" width="815">

**파이프라인 위치** · ① 데이터 준비 → ⑥ 평가

> 🎯 **상품 13개 CSV에서 크기 · 조건 상품 · 가장 비싼 상품 · 가장 많은 카테고리를 숫자로 답하려면?**

💡 **쉽게 말하면**

- 학생 표 대신 상품 표 · 쓰는 방법은 똑같음.
- 질문마다 어느 장의 방법인지 먼저 고르고 실행.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>읽기 · 크기 · 열 · 자료형</td><td>1-1 · 1-2</td></tr>
<tr><td>2</td><td>필요한 열만</td><td>2-1</td></tr>
<tr><td>3</td><td>가격 20,000 이상 &amp; 평점 4.3 이상</td><td>3-2</td></tr>
<tr><td>4</td><td>평균 가격 · 최고 평점 · 재고 부족 열</td><td>4-1 · 4-2</td></tr>
<tr><td>5</td><td>재고 15 이하를 비싼 순 · 카테고리 개수</td><td>4-3 · 4-4</td></tr>
</tbody></table>

**먼저 예측** · 가격 20,000원 이상이면서 평점 4.3 이상인 상품은 몇 개? — 답은 코드 실행 결과 아래에

In [12]:
# 처음 보는 작은 CSV: 확인 → 선택
# 왜 · 새 파일은 크기 · 열 이름 · 자료형부터 확인한 뒤 고르기 위해
products = pd.read_csv("data/products_basic.csv")                         # 상품 13개
print("shape:", products.shape, "/ columns:", products.columns.tolist())  # 크기 · 열 이름
print(products.dtypes.astype(str).to_dict())                              # 열마다 자료형
print(products[["product", "price", "rating"]].head(3))                   # Gate 1: 필요한 열만


# 조건 · 계산 · 새 열
# 왜 · 앞에서 배운 조건 · 계산을 실제 파일에 이어 쓰기 위해
gate2 = products[(products["price"] >= 20000) & (products["rating"] >= 4.3)]  # Gate 2: 두 조건
print("Gate 2:", gate2["product"].tolist())                                   # 상품 이름
# 통계
print("평균 가격:", round(products["price"].mean()), "/ 최고 평점:", products["rating"].max())
products["low_stock"] = products["stock"] <= 10                               # 재고 10개 이하
print("low_stock:", products["low_stock"].value_counts().to_dict())           # True · False 개수

shape: (13, 5) / columns: ['product', 'category', 'price', 'stock', 'rating']
{'product': 'str', 'category': 'str', 'price': 'int64', 'stock': 'int64', 'rating': 'float64'}
         product  price  rating
0    무선 마우스  28000     4.3
1  기계식 키보드  89000     4.7
2       USB 허브  35000     4.1
Gate 2: ['무선 마우스', '기계식 키보드', '블루투스 스피커', '텀블러', '티셔츠', '후드']
평균 가격: 33615 / 최고 평점: 4.8
low_stock: {False: 9, True: 4}


**실행 흐름**

1. 재고 15개 이하 상품을 비싼 순으로 정렬 · 출력
2. `nlargest()`로 가장 비싼 3개
3. 카테고리별 상품 수와 1위

In [13]:
# 정렬 · 개수
# 왜 · 결과를 기준대로 줄 세우고 개수로 마무리 확인하기 위해
gate3 = products[products["stock"] <= 15].sort_values("price", ascending=False)  # Gate 3
print("Gate 3:", gate3["product"].tolist())                                 # 비싼 순
print("가장 비싼 3개:", products.nlargest(3, "price")["product"].tolist())  # 정렬 + 앞 3개
counts = products["category"].value_counts()                                # 카테고리별 상품 수
print("카테고리:", counts.to_dict(), "/ 1위:", counts.idxmax())             # 가장 많은 카테고리

Gate 3: ['기계식 키보드', '후드', '블루투스 스피커', '데스크 램프', 'USB 허브', '티셔츠', '텀블러']
가장 비싼 3개: ['기계식 키보드', '후드', '블루투스 스피커']
카테고리: {'전자기기': 4, '문구': 3, '생활': 3, '패션': 3} / 1위: 전자기기


**결과** · 예측 정답은 6개

- `shape: (13, 5)` — 상품 13개 · 열 5개
- `'price': 'int64'` — 가격은 정수 → 계산 가능
- `Gate 2: ['무선 마우스', '기계식 키보드', '블루투스 스피커', '텀블러', '티셔츠', '후드']` — 두 조건 모두 만족 6개
- `평균 가격: 33615 / 최고 평점: 4.8` — 437000 ÷ 13 = 33615 · 최고 평점은 후드
- `low_stock: {False: 9, True: 4}` — 재고 10개 이하 4개
- `Gate 3: ['기계식 키보드', '후드', '블루투스 스피커', '데스크 램프', 'USB 허브', '티셔츠', '텀블러']` — 재고 15 이하 7개를 비싼 순으로
- `가장 비싼 3개: ['기계식 키보드', '후드', '블루투스 스피커']` — `nlargest(3, "price")`
- `1위: 전자기기` — 카테고리 상품 수 1위 (4개)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df[조건1 &amp; 조건2]</code></td><td>두 조건 모두</td><td>Gate 2처럼 기준이 둘일 때</td></tr>
<tr><td>조건 → <code>sort_values</code></td><td>고른 뒤 줄 세우기</td><td>Gate 3</td></tr>
<tr><td><code>nlargest(n, 열)</code></td><td>정렬 + 앞 n개</td><td>상위 n개만</td></tr>
<tr><td><code>value_counts().idxmax()</code></td><td>가장 많은 값의 이름</td><td>최빈 범주</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `idxmax()`는 동점이면 먼저 나온 하나만 → 개수 표도 함께 보기

> ✅ **한 줄 정리** · 처음 보는 표도 크기 · 열 확인 → 열 선택 → 조건 → 계산 → 정렬 · 개수 순서로 질문에 숫자로 답한다.

**딥러닝 코드에서는**

- 새 데이터셋의 첫 30분이 이 흐름 — 크기 · 라벨 분포 · 이상한 행 확인

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 표 확인</td><td><code>(6, 4)</code> · 열 이름 · 자료형 <code>int64</code> / <code>str</code></td><td>1-1 · 1-2</td></tr>
<tr><td>2. 고르기</td><td>한 열 Series <code>(6,)</code> · 두 열 DataFrame <code>(6, 2)</code> · 정렬 뒤 <code>loc[0]</code> 민수 vs <code>iloc[0]</code> 지수</td><td>2-1 · 2-2</td></tr>
<tr><td>3. 조건</td><td>80점 이상 4명 · 부산 &amp; 85↑ 지수 · 현우 · 괄호 없으면 TypeError</td><td>3-1 · 3-2</td></tr>
<tr><td>4. 계산 · 정렬 · 개수</td><td>평균 83.5 · 할인 금액 5000 · 동점은 두 번째 기준 · 합격 4명</td><td>4장 · 5</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>첫 확인</td><td><code>shape</code> · <code>dtypes</code> · <code>head</code></td><td><code>df.info()</code></td></tr>
<tr><td><code>A</code></td><td>열 고르기</td><td><code>df["a"]</code> Series · <code>df[["a"]]</code> 표</td><td><code>X = df[cols]</code></td></tr>
<tr><td><code>A</code></td><td>loc · iloc</td><td>라벨 · 끝 포함 vs 위치 · 끝 미포함</td><td><code>df.iloc[:800]</code></td></tr>
<tr><td><code>A</code></td><td>조건</td><td>mask → <code>df[mask]</code> · 괄호 + <code>&amp;</code></td><td><code>df[df["split"] == "train"]</code></td></tr>
<tr><td><code>A</code></td><td>새 열 · 개수</td><td>같은 행끼리 계산 · <code>value_counts</code></td><td>경로 열 · 클래스 분포</td></tr>
<tr><td><code>B</code></td><td>통계 · 정렬</td><td><code>agg</code> · <code>sort_values([...])</code></td><td>실험 결과 정렬</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 표의 (행 수, 열 수)는 무엇으로 확인하는가?
- **1-2** · 숫자 열인지 문자열 열인지는 무엇으로 보는가?
- **2-1** · 한 열과 두 열을 고른 결과의 type은?
- **2-2** · 정렬 뒤 `loc[0]`과 `iloc[0]`은 무엇이 다른가?
- **3-1** · 조건으로 행을 고르는 두 단계는?
- **3-2** · 조건 두 개를 묶는 법은?
- **4-2** · 새 열은 어떻게 만드는가?
- **4-4** · 값별 개수와 비율은?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `students`에서 서울이 **아닌** 학생 이름 목록 출력 (`~` 사용)
2. `students`의 점수 상위 2명 이름 출력 (`nlargest`)

In [14]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 3-2의 조건을 '대전 또는 85점 이상'으로 바꾸면 몇 명이고 누구인가?
2. 4-2에서 `sales["discount"] = [0.1, 0.2]`처럼 2개만 넣으면?
3. 2-2에서 `ranked.iloc[0:2]`와 `ranked.loc[1:4]`는 각각 누구인가?

In [15]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · `df.shape`
- **1-2** · `df.dtypes` (pandas 3은 문자열이 `str`)
- **2-1** · Series · DataFrame
- **2-2** · 라벨 0인 행 vs 지금 첫 줄
- **3-1** · 조건 → True/False(mask) → `df[mask]`
- **3-2** · 괄호 + `&amp;`(그리고) · `|`(또는)
- **4-2** · `df["새열"] = 열 계산` (같은 행끼리)
- **4-4** · `value_counts()` · `value_counts(normalize=True)`

#### 코드 문제 답

```python
print(students[~(students["city"] == "서울")]["name"].tolist())  # ['지수', '철수', '현우']
print(students.nlargest(2, "score")["name"].tolist())  # ['지수', '유진']
```

#### 변형 문제 답

*문제 → 답*

- **1** · 4명 · 지수(95) · 철수(대전) · 유진(91) · 현우(88) (표 순서)
- **2** · ValueError — 3행에 값 2개 (길이 불일치)
- **3** · 둘 다 지수 · 유진 — `iloc[0:2]`는 첫 두 줄, `loc[1:4]`는 지금 줄 순서에서 라벨 1(지수)부터 라벨 4(유진)까지

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 정렬 뒤 <code>loc[0]</code>을 1등으로 답함</td><td>라벨과 위치를 같다고 봄</td><td>첫 줄은 <code>iloc[0]</code>, <code>loc</code>은 라벨</td><td>지금 찾는 게 이름표인가 자리인가?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 새 CSV를 받으면 무엇부터 확인하나요?**
> - shape로 행과 열 수를 보고, dtypes로 숫자 열과 문자열 열을 구분합니다.
> - head와 tail로 값 형식과 끝부분의 이상한 행을 확인한 뒤 분석을 시작합니다.

**Q2. loc과 iloc의 차이는요?**
> - loc은 라벨 기준이라 슬라이스의 끝 라벨을 포함하고, iloc은 위치 기준이라 끝 위치를 포함하지 않습니다.
> - 정렬이나 필터 뒤에는 라벨과 위치가 달라지므로 첫 줄이 필요하면 iloc[0]을 씁니다.

**Q3. 조건 두 개로 행을 고를 때 주의할 점은요?**
> - and 대신 행마다 계산하는 &와 |를 쓰고, &가 비교보다 먼저 계산되므로 조건마다 괄호를 칩니다.
> - 같은 열의 여러 값은 isin으로 짧게 씁니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>pd.DataFrame({열: 목록})</code></td><td>pandas</td><td>사전 → 표</td><td>1-1 <code>students = pd.DataFrame({</code></td></tr>
<tr><td><code>.shape</code> · <code>.columns</code> · <code>.size</code></td><td>DataFrame 속성</td><td>크기 · 열 이름 · 칸 수</td><td>1-1 <code>students.shape</code></td></tr>
<tr><td><code>.dtypes</code> · <code>.head()</code> · <code>.tail()</code></td><td>DataFrame</td><td>자료형 · 앞뒤 일부</td><td>1-2 <code>students.tail(3)</code></td></tr>
<tr><td><code>df["열"]</code> · <code>df[["열1", "열2"]]</code></td><td>DataFrame 인덱싱</td><td>Series · DataFrame으로 열 고르기</td><td>2-1 <code>students[["name", "city"]]</code></td></tr>
<tr><td><code>.loc[]</code> · <code>.iloc[]</code></td><td>DataFrame 인덱서</td><td>라벨(끝 포함) · 위치(끝 미포함)</td><td>2-2 <code>students.iloc[:2, :2]</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>df[mask]</code></td><td>DataFrame 인덱싱</td><td>True 행 고르기</td><td>3-1 <code>students[mask]</code></td></tr>
<tr><td><code>&amp;</code> · <code>&#124;</code> · <code>.isin()</code></td><td>pandas 연산자 · Series 메서드</td><td>그리고 · 또는 · 여러 값 중 하나</td><td>3-2 <code>(students["city"] == "부산") &amp; (...)</code></td></tr>
<tr><td><code>.mean()</code> · <code>.sum()</code> · <code>.agg()</code></td><td>Series · DataFrame 메서드</td><td>통계</td><td>4-1 <code>s.mean()</code></td></tr>
<tr><td><code>df["새열"] = ...</code></td><td>DataFrame 할당</td><td>새 열</td><td>4-2 <code>sales["discount_amount"] = ...</code></td></tr>
<tr><td><code>.sort_values()</code> · <code>.nlargest()</code></td><td>DataFrame 메서드</td><td>정렬 · 상위 n개</td><td>4-3 <code>sort_values(["study_hours", "score"])</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>.value_counts()</code> · <code>.idxmax()</code></td><td>Series 메서드</td><td>값별 개수 · 1위 이름</td><td>4-4 <code>students["passed"].value_counts()</code></td></tr>
<tr><td><code>pd.read_csv()</code></td><td>pandas</td><td>CSV → 표</td><td>5 <code>pd.read_csv("data/products_basic.csv")</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>읽자마자 <code>shape</code> · <code>dtypes</code> · <code>head</code></td><td>출력 없이 바로 계산</td></tr>
<tr><td>조건마다 괄호 + <code>&amp;</code></td><td>Series에 <code>and</code> · <code>or</code></td></tr>
<tr><td>정렬 뒤 첫 줄은 <code>iloc[0]</code></td><td>정렬 뒤 <code>loc[0]</code>을 1등으로</td></tr>
<tr><td>동점은 두 번째 기준 열</td><td>기준 하나로 순서를 믿기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>df["a"]</code> vs <code>df[["a"]]</code></td><td>Series vs DataFrame</td><td>표 모양 유지는 이중 대괄호</td></tr>
<tr><td><code>loc</code> vs <code>iloc</code></td><td>라벨 · 끝 포함 vs 위치 · 끝 미포함</td><td>정렬 뒤 첫 줄은 <code>iloc[0]</code></td></tr>
<tr><td><code>&amp;</code> vs <code>and</code></td><td>행마다 vs 값 하나</td><td><code>&amp;</code> + 괄호</td></tr>
<tr><td><code>sort_values</code> vs <code>nlargest</code></td><td>전체 정렬 vs 상위 n개</td><td>상위 n개만이면 <code>nlargest</code></td></tr>
<tr><td><code>value_counts()</code> vs <code>normalize=True</code></td><td>개수 vs 비율</td><td>불균형 확인은 비율</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · loc/iloc 그림을 가리고 결과 말하기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>5장 종합 실습을 빈 셀에 직접 다시 짜기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 전체 정리 1 ~ 4장 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `00_pandas_foundations.ipynb`의 설명 · 예측 질문 · 코드 셀 · 빈 코드 칸 13개 · 글 빈칸
- 빈 코드 칸 13개는 모두 직접 풀어 실행으로 확인 · 코드의 `# 빈칸 00-①` ~ `⑬`
- 원본 1 ~ 9장을 4개 장 · 10개 섹션으로 묶음 · 원본 마지막 말(처음 보는 작은 CSV)을 5장 종합 실습으로
- 원본의 접힌 힌트(details 태그)는 0장 '원본 힌트' 목록으로 옮김 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>1 · 2장 불러오기 · 첫 데이터 · shape · columns · dtypes · head · tail</td><td>빈칸 00-① · ②</td><td>1-1 · 1-2</td></tr>
<tr><td>3장 열 선택 · Series</td><td>빈칸 00-③ · ④</td><td>2-1</td></tr>
<tr><td>4장 loc · iloc</td><td>빈칸 00-⑤</td><td>2-2</td></tr>
<tr><td>5장 Boolean Filtering · 글 빈칸</td><td>빈칸 00-⑥ ~ ⑨</td><td>3-1 · 3-2</td></tr>
<tr><td>6장 계산 · passed 열</td><td>빈칸 00-⑩</td><td>4-1 · 4-2</td></tr>
<tr><td>7장 작은 판매 데이터</td><td>빈칸 00-⑪</td><td>4-2</td></tr>
<tr><td>8장 정렬</td><td>빈칸 00-⑫</td><td>4-3</td></tr>
<tr><td>9장 값의 개수</td><td>빈칸 00-⑬</td><td>4-4</td></tr>
<tr><td>Foundations 정리 · 가장 중요한 흐름</td><td>질문 → 방법 표</td><td>그래서 오늘 무엇을 배웠는가? · 최종 요약표</td></tr>
<tr><td>(다음 단계 안내)</td><td>처음 보는 작은 CSV</td><td>5 종합 실습</td></tr>
</tbody></table>

### 전체 코드

1 ~ 5장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 표 만들기 · 크기 · 열 이름
2. 1-2 자료형 · `head` · `tail`
3. 블록마다 결과 출력

In [16]:
# ---------- 1-1 ----------
# 학생 6명 표 만들기: {열 이름: 값 목록}
# 왜 · 파일 없이도 결과를 눈으로 맞춰 볼 수 있는 작은 표를 직접 만들기 위해
students = pd.DataFrame({  # 사전의 키 = 열 이름, 값 목록 = 그 열의 값
    "name": ["민수", "지수", "영희", "철수", "유진", "현우"],
    "score": [78, 95, 84, 65, 91, 88],
    "city": ["서울", "부산", "서울", "대전", "서울", "부산"],
    "study_hours": [3, 7, 5, 2, 6, 5],
})
print(students)            # 표 전체를 눈으로 확인


# 빈칸 00-① 크기와 열 이름
# 왜 · 처음 보는 표는 크기와 열 이름부터 확인해 잘못 읽은 곳을 찾기 위해
print("shape:", students.shape)                         # (행 수, 열 수)
print("columns:", students.columns.tolist())            # 열 이름 목록
print("len:", len(students), "/ size:", students.size)  # 행 수 · 칸 수


# ---------- 1-2 ----------
# 열마다 자료형
# 왜 · 숫자로 계산할 열이 문자열로 읽히지 않았는지 확인하기 위해
print(students.dtypes)  # 숫자 열인가 문자열 열인가


# 처음 3행 · 원본 tail(2)
# 왜 · 전체를 출력하지 않고 앞 · 뒤 일부로 내용을 확인하기 위해
print(students.head(3))  # 기본값 5 → 3으로 지정
print(students.tail(2))  # 마지막 2행


# 빈칸 00-② 마지막 3행
print(students.tail(3))  # 3 · 4 · 5번 행

   name  score  city  study_hours
0  민수     78  서울            3
1  지수     95  부산            7
2  영희     84  서울            5
3  철수     65  대전            2
4  유진     91  서울            6
5  현우     88  부산            5
shape: (6, 4)
columns: ['name', 'score', 'city', 'study_hours']
len: 6 / size: 24
name             str
score          int64
city             str
study_hours    int64
dtype: object
   name  score  city  study_hours
0  민수     78  서울            3
1  지수     95  부산            7
2  영희     84  서울            5
   name  score  city  study_hours
4  유진     91  서울            6
5  현우     88  부산            5
   name  score  city  study_hours
3  철수     65  대전            2
4  유진     91  서울            6
5  현우     88  부산            5


**실행 흐름**

1. 2-1 열 고르기 (Series vs DataFrame)
2. 2-2 `loc` · `iloc`
3. 정렬 뒤 라벨 · 위치 차이 확인

In [17]:
# ---------- 2-1 ----------
# 한 열 → Series
# 왜 · 대괄호 하나로 고른 결과가 1차원 Series라는 것을 확인하기 위해
score = students["score"]                             # 열 이름 하나
print(type(students).__name__, type(score).__name__)  # 표 vs 한 줄 목록
print(score.tolist())                                 # 점수 6개


# 빈칸 00-③ city 한 열 · name과 city 두 열
# 왜 · 괄호 하나와 이중 괄호의 결과 모양 차이를 비교하기 위해
city = students["city"]                 # Series
name_city = students[["name", "city"]]  # 열 이름 목록 → DataFrame
print(name_city.head(2))                # 두 열이 표로 나옴


# 빈칸 00-④ 두 결과의 type · shape
# 1차원 vs 2차원
print(type(city).__name__, city.shape, "/", type(name_city).__name__, name_city.shape)


# ---------- 2-2 ----------
# loc: 라벨 기준 (끝 라벨 포함) · iloc: 위치 기준 (끝 위치 미포함)
# 왜 · 같아 보이는 두 방법이 끝 포함 여부에서 다르다는 것을 확인하기 위해
print(students.loc[0, "name"], students.iloc[0, 0])  # 지금은 라벨 = 위치라 같음
print(students.loc[0:2, ["name", "score"]].shape)    # 라벨 0 · 1 · 2 → 3행
print(students.iloc[0:3, 0:2].shape)                 # 위치 0 · 1 · 2 → 3행


# 빈칸 00-⑤ 첫 학생 · 처음 3명 이름 · 점수 · 첫 2행 2열
# 왜 · 행 · 열을 함께 고르는 대표 모양을 손에 익히기 위해
print(students.loc[0].to_dict())             # 라벨 0 행 전체
print(students.loc[0:2, ["name", "score"]])  # 처음 3명
print(students.iloc[:2, :2])                 # 첫 2행 · 첫 2열


# 정렬하면 라벨과 위치가 달라짐
# 왜 · 정렬 뒤에는 loc와 iloc가 다른 행을 가리킨다는 함정을 보이기 위해
ranked = students.sort_values("score", ascending=False)  # 라벨은 사람을 따라감
print("라벨 순서:", ranked.index.tolist())               # 첫 줄 라벨은 1
print("loc[0] →", ranked.loc[0, "name"], "/ iloc[0] →", ranked.iloc[0]["name"])  # 라벨 0 vs 첫 줄

DataFrame Series
[78, 95, 84, 65, 91, 88]
   name  city
0  민수  서울
1  지수  부산
Series (6,) / DataFrame (6, 2)
민수 민수
(3, 2)
(3, 2)
{'name': '민수', 'score': 78, 'city': '서울', 'study_hours': 3}
   name  score
0  민수     78
1  지수     95
2  영희     84
   name  score
0  민수     78
1  지수     95
라벨 순서: [1, 4, 5, 2, 0, 3]
loc[0] → 민수 / iloc[0] → 지수


**실행 흐름**

1. 3-1 조건 mask로 행 고르기
2. 3-2 `&` · `|` · `isin`
3. 괄호 누락 에러 확인

In [18]:
# ---------- 3-1 ----------
# 조건 → 행마다 True/False (mask)
# 왜 · 조건식이 행마다 True/False를 만든다는 것을 먼저 눈으로 보기 위해
mask = students["score"] >= 80          # 6개 답
print("mask:", mask.tolist())           # 민수 · 철수만 False
print(students[mask]["name"].tolist())  # True 행의 이름


# 빈칸 00-⑥ · ⑦ 85점 이상 · 부산 학생
# 왜 · mask를 대괄호에 넣어 조건에 맞는 행만 고르기 위해
print("85점 이상:", students[students["score"] >= 85]["name"].tolist())  # 조건 → 고르기
print("부산:", students[students["city"] == "부산"]["name"].tolist())    # 문자열 비교
print("85점 이상 몇 명:", (students["score"] >= 85).sum())               # True = 1로 세기


# ---------- 3-2 ----------
# 원본 예시: 서울 그리고 80점 이상 · 서울 또는 부산
# 왜 · 두 조건을 &와 |로 묶는 방법을 비교하기 위해
seoul_80 = (students["city"] == "서울") & (students["score"] >= 80)        # 조건마다 괄호 + &
seoul_busan = (students["city"] == "서울") | (students["city"] == "부산")  # 또는 = |
print("서울 & 80↑:", students[seoul_80]["name"].tolist())                  # 둘 다 True
print("서울 | 부산:", students[seoul_busan]["name"].tolist())              # 하나라도 True


# 빈칸 00-⑧ 부산 & 85점 이상 · ⑨ 서울 또는 부산 (isin)
# 왜 · 값 여러 개 중 하나인지 볼 때 isin이 | 보다 짧다는 것을 보이기 위해
busan_85 = (students["city"] == "부산") & (students["score"] >= 85)  # 그리고
print("부산 & 85↑:", students[busan_85]["name"].tolist())            # 지수 · 현우
# | 여러 번 = isin
print("isin 같은가:", students["city"].isin(["서울", "부산"]).equals(seoul_busan))


# 괄호를 빼면 & 가 먼저 계산됨
# 왜 · 연산자 우선순위 때문에 조건마다 괄호가 필요하다는 것을 에러로 확인하기 위해
try:
    students[students["city"] == "서울" & students["score"] >= 80]  # "서울" & score 부터
except TypeError:                                                   # 문자열과 & 계산 불가
    print("괄호 없음 → TypeError")                                  # 괄호가 필요한 이유

mask: [False, True, True, False, True, True]
['지수', '영희', '유진', '현우']
85점 이상: ['지수', '유진', '현우']
부산: ['지수', '현우']
85점 이상 몇 명: 3
서울 & 80↑: ['영희', '유진']
서울 | 부산: ['민수', '지수', '영희', '유진', '현우']
부산 & 85↑: ['지수', '현우']
isin 같은가: True
괄호 없음 → TypeError


**실행 흐름**

1. 4-1 통계 · `agg`
2. 4-2 새 열 만들기
3. 4-3 정렬 · 4-4 `value_counts`

In [19]:
# ---------- 4-1 ----------
# 원본: 점수의 평균 · 최댓값 · 최솟값 · 합계
# 왜 · 반복문 없이 열 전체 통계를 바로 구하기 위해
s = students["score"]  # 한 열 = Series
print("평균:", s.mean(), "/ 최대:", s.max(), "/ 최소:", s.min(), "/ 합계:", s.sum())  # 계산 버튼


# 빈칸 00-⑩ 공부 시간 평균 · 최댓값 · 합계
h = students["study_hours"]  # 공부 시간 열
# 소수 둘째 자리
print("공부 시간 평균:", round(h.mean(), 2), "/ 최대:", h.max(), "/ 합계:", h.sum())


# 여러 통계를 한 번에
# 왜 · 여러 열 · 여러 통계를 agg 한 번으로 표로 받기 위해
print(students[["score", "study_hours"]].agg(["mean", "max", "sum"]).round(2))  # 행 = 통계


# ---------- 4-2 ----------
# 원본: 80점 이상이면 True인 passed 열
# 왜 · 비교 결과를 새 열로 저장해 뒤에서 개수 · 비율을 세기 위해
students["passed"] = students["score"] >= 80  # 비교 결과를 새 열로
print(students.shape)                         # 열이 5개가 됨


# 원본 판매 데이터: 가격 × 수량 → sales 열
# 왜 · 같은 행끼리 계산한 값을 새 열로 만드는 방법을 보이기 위해
sales = pd.DataFrame({"product": ["키보드", "마우스", "노트"],  # 상품 3개
                      "price": [50000, 25000, 3000],
                      "quantity": [1, 2, 3]})
sales["sales"] = sales["price"] * sales["quantity"]             # 같은 행끼리 곱
sales["discount"] = [0.1, 0.0, 0.2]                             # 행 수와 같은 길이의 목록


# 빈칸 00-⑪ 할인 금액 = 할인 전 금액 × 할인율
# 왜 · 앞에서 만든 열을 다시 계산에 쓰는 흐름을 연습하기 위해
sales["discount_amount"] = sales["sales"] * sales["discount"]  # 새 열
print(sales)                                                   # 열 6개 확인


# ---------- 4-3 ----------
# 원본: 점수 오름차순 · 내림차순
# 왜 · ascending 하나로 정렬 방향이 바뀌는 것을 비교하기 위해
asc = students.sort_values("score")                    # 작은 값부터
desc = students.sort_values("score", ascending=False)  # 큰 값부터
print("오름차순:", asc["name"].tolist())               # 65 → 95
print("내림차순:", desc["name"].tolist())              # 95 → 65


# 빈칸 00-⑫ 공부 시간이 긴 학생부터 (같으면 점수 높은 순)
# 왜 · 동점일 때 순서를 정하려고 두 번째 기준 열을 함께 주기 위해
by_hours = students.sort_values(["study_hours", "score"], ascending=False)  # 동점은 두 번째 기준
print(by_hours[["name", "study_hours", "score"]])                           # 정렬 결과


# ---------- 4-4 ----------
# 원본: 도시별 학생 수
# 왜 · 범주마다 몇 개인지 한 줄로 세기 위해
print(students["city"].value_counts().to_dict())  # 값별 개수 (많은 순)


# 빈칸 00-⑬ 합격 · 불합격 학생 수
# 왜 · 개수뿐 아니라 비율(normalize)도 함께 보기 위해
print("passed:", students["passed"].value_counts().to_dict())  # True 4 · False 2
print("비율:", students["passed"].value_counts(normalize=True).round(2).to_dict())  # 개수 대신 비율

평균: 83.5 / 최대: 95 / 최소: 65 / 합계: 501
공부 시간 평균: 4.67 / 최대: 7 / 합계: 28
      score  study_hours
mean   83.5         4.67
max    95.0         7.00
sum   501.0        28.00
(6, 5)
  product  price  quantity  sales  discount  discount_amount
0  키보드  50000         1  50000       0.1           5000.0
1  마우스  25000         2  50000       0.0              0.0
2    노트   3000         3   9000       0.2           1800.0
오름차순: ['철수', '민수', '영희', '현우', '유진', '지수']
내림차순: ['지수', '유진', '현우', '영희', '민수', '철수']
   name  study_hours  score
1  지수            7     95
4  유진            6     91
5  현우            5     88
2  영희            5     84
0  민수            3     78
3  철수            2     65
{'서울': 3, '부산': 2, '대전': 1}
passed: {True: 4, False: 2}
비율: {True: 0.67, False: 0.33}


**실행 흐름**

1. CSV 읽고 크기 · 열 확인
2. 조건 · 계산 · 새 열
3. 통계 · 정렬 · 개수

In [20]:
# ---------- 5. 종합 실습 ----------
# 처음 보는 작은 CSV: 확인 → 선택
# 왜 · 새 파일은 크기 · 열 이름 · 자료형부터 확인한 뒤 고르기 위해
products = pd.read_csv("data/products_basic.csv")                         # 상품 13개
print("shape:", products.shape, "/ columns:", products.columns.tolist())  # 크기 · 열 이름
print(products.dtypes.astype(str).to_dict())                              # 열마다 자료형
print(products[["product", "price", "rating"]].head(3))                   # Gate 1: 필요한 열만


# 조건 · 계산 · 새 열
# 왜 · 앞에서 배운 조건 · 계산을 실제 파일에 이어 쓰기 위해
gate2 = products[(products["price"] >= 20000) & (products["rating"] >= 4.3)]  # Gate 2: 두 조건
print("Gate 2:", gate2["product"].tolist())                                   # 상품 이름
# 통계
print("평균 가격:", round(products["price"].mean()), "/ 최고 평점:", products["rating"].max())
products["low_stock"] = products["stock"] <= 10                               # 재고 10개 이하
print("low_stock:", products["low_stock"].value_counts().to_dict())           # True · False 개수


# 정렬 · 개수
# 왜 · 결과를 기준대로 줄 세우고 개수로 마무리 확인하기 위해
gate3 = products[products["stock"] <= 15].sort_values("price", ascending=False)  # Gate 3
print("Gate 3:", gate3["product"].tolist())                                 # 비싼 순
print("가장 비싼 3개:", products.nlargest(3, "price")["product"].tolist())  # 정렬 + 앞 3개
counts = products["category"].value_counts()                                # 카테고리별 상품 수
print("카테고리:", counts.to_dict(), "/ 1위:", counts.idxmax())             # 가장 많은 카테고리

shape: (13, 5) / columns: ['product', 'category', 'price', 'stock', 'rating']
{'product': 'str', 'category': 'str', 'price': 'int64', 'stock': 'int64', 'rating': 'float64'}
         product  price  rating
0    무선 마우스  28000     4.3
1  기계식 키보드  89000     4.7
2       USB 허브  35000     4.1
Gate 2: ['무선 마우스', '기계식 키보드', '블루투스 스피커', '텀블러', '티셔츠', '후드']
평균 가격: 33615 / 최고 평점: 4.8
low_stock: {False: 9, True: 4}
Gate 3: ['기계식 키보드', '후드', '블루투스 스피커', '데스크 램프', 'USB 허브', '티셔츠', '텀블러']
가장 비싼 3개: ['기계식 키보드', '후드', '블루투스 스피커']
카테고리: {'전자기기': 4, '문구': 3, '생활': 3, '패션': 3} / 1위: 전자기기
